# Energy to Probability

Chapter 17: Score Function and Energy-Based Models · *Mathematics for AI and Machine Learning* by Fuheng Wu

Convert a customizable 1D energy landscape to normalized density.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-17/energy-to-probability/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

# The demo's defaults: E(x) = (x^2 - w^2)^2 / 4 + tilt x, p(x) ∝ exp(-beta E(x)) on [-3, 3]
beta, tilt, well = 1.5, 0.0, 1.0
x = torch.linspace(-3, 3, 30001, dtype=torch.float64)


def density(beta, tilt, well):
    E = 0.25 * (x ** 2 - well ** 2) ** 2 + tilt * x
    q = torch.exp(-beta * E)
    Z = torch.trapezoid(q, x)                    # partition function
    return q / Z, Z


p, Z = density(beta, tilt, well)
print(f"Z = {Z.item():.4f}  (the demo estimates it on the same interval)")
print("density integrates to", torch.trapezoid(p, x).item())
print(f"P(x > 0) = {torch.trapezoid(p[x > 0], x[x > 0]).item():.4f}")

# ── Temperature: large beta concentrates the mass at the energy minima ──────
print("\n beta    P(|x| within 0.25 of a well)")
near = ((x.abs() - well).abs() < 0.25)
for b in (0.2, 0.5, 1.5, 5.0, 20.0):
    pb, _ = density(b, tilt, well)
    print(f"{b:5.1f}   {torch.trapezoid(pb * near, x).item():.4f}")

# ── Tilt breaks the symmetry: probabilities follow e^{-beta * energy difference}
print("\n tilt   P(right well, x > 0)")
for t in (0.0, 0.1, 0.3, -0.3):
    pt, _ = density(beta, t, well)
    print(f"{t:+.1f}   {torch.trapezoid(pt[x > 0], x[x > 0]).item():.4f}")

Z = 2.7436  (the demo estimates it on the same interval)
density integrates to 1.0000000000000002
P(x > 0) = 0.4999

 beta    P(|x| within 0.25 of a well)
  0.2   0.2320
  0.5   0.2784
  1.5   0.3532
  5.0   0.5053
 20.0   0.8433

 tilt   P(right well, x > 0)
+0.0   0.4999
+0.1   0.4365
+0.3   0.3173
-0.3   0.6827
